In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.5 MB/s eta 0:00:00:00:0100:01


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "I love Artificial Intelligence",
    "Machine Learning is amazing",
    "Deep Learning uses neural networks"
]

embeddings = model.encode(sentences)

print("Embedding Shape:", embeddings.shape)
print(embeddings[0])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (3, 384)
[-2.61358768e-02 -6.41851872e-02  5.79342954e-02 -2.27682386e-02
  3.99345867e-02 -6.94327131e-02  5.09679951e-02  2.81137675e-02
  5.69299720e-02  4.83488739e-02 -4.29283865e-02  6.80153258e-03
  3.28423874e-03  4.65277582e-02 -3.27342041e-02  2.20695809e-02
 -4.11612764e-02 -1.76253356e-02 -1.33416310e-01 -1.12813167e-01
 -9.07842219e-02  4.84751500e-02  2.20636446e-02 -4.63822559e-02
 -2.57994588e-02  3.99872102e-02  2.86494009e-03 -6.62114024e-02
 -8.65888037e-03 -8.70667845e-02 -2.54888553e-03  3.31528969e-02
  7.47497305e-02  3.48980277e-04 -6.45887405e-02  5.41341789e-02
  3.39241605e-03 -1.88701767e-02  6.48413077e-02  2.78313411e-04
 -3.98724824e-02 -1.17227212e-02  5.26452921e-02 -5.42398263e-03
  4.97092754e-02  7.94329196e-02 -7.28190169e-02 -4.01672684e-02
  7.00027496e-02  4.89695333e-02 -8.06456581e-02 -6.57426892e-03
 -2.13148128e-02  8.11337680e-03  6.76513137e-03  2.71840636e-02
  5.28749451e-02  1.98686644e-02 -2.27795262e-02 -6.46123439e-02

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Artificial Intelligence is transforming the world"

tokens = tokenizer.tokenize(text)

token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(tokens)
print(token_ids)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

['artificial', 'intelligence', 'is', 'transforming', 'the', 'world']
[7976, 4454, 2003, 17903, 1996, 2088]


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "I love AI",
    "Artificial Intelligence is amazing"
]

embeddings = model.encode(sentences)

similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print(similarity)

[[0.7304046]]


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Tuple

class SimpleRAG:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print("Loading embedding model...")
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None
    
    def add_documents(self, docs: List[str]):
        self.documents.extend(docs)
        print(f"Added {len(docs)} documents. Total: {len(self.documents)}")
        
        self.embeddings = self.model.encode(self.documents, normalize_embeddings=True)
        print("Embeddings created successfully!")
    
    def cosine_similarity(self, query_embedding: np.ndarray) -> np.ndarray:
        similarities = np.dot(self.embeddings, query_embedding)
        return similarities
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        if self.embeddings is None:
            raise ValueError("No documents added yet!")
        
        query_embedding = self.model.encode(query, normalize_embeddings=True)
        
        similarities = self.cosine_similarity(query_embedding)
        
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            results.append((self.documents[idx], float(similarities[idx])))
        
        return results

if __name__ == "__main__":
    rag = SimpleRAG()
    
    documents = [
        "Python is a popular programming language known for its simplicity.",
        "Machine Learning is a subset of artificial intelligence.",
        "RAG stands for Retrieval Augmented Generation.",
        "Cosine similarity measures the similarity between two vectors.",
        "Large Language Models are trained on massive amounts of text data."
    ]
    
    rag.add_documents(documents)
    
    query = "What is RAG in AI?"
    
    print(f"\n🔍 Query: {query}\n")
    results = rag.retrieve(query, top_k=3)
    
    for i, (doc, score) in enumerate(results, 1):
        print(f"{i}. Similarity: {score:.4f}")
        print(f"   {doc}\n")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Added 5 documents. Total: 5
Embeddings created successfully!

🔍 Query: What is RAG in AI?

1. Similarity: 0.6456
   RAG stands for Retrieval Augmented Generation.

2. Similarity: 0.3270
   Machine Learning is a subset of artificial intelligence.

3. Similarity: 0.1663
   Python is a popular programming language known for its simplicity.



In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean, cityblock
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = ["I love AI", "Artificial Intelligence is amazing"]

embeddings = model.encode(sentences)

cos_sim = cosine_similarity([embeddings[0]], [embeddings[1]])

euc_dist = euclidean(embeddings[0], embeddings[1])

man_dist = cityblock(embeddings[0], embeddings[1])

dot_product = np.dot(embeddings[0], embeddings[1])

print("Cosine Similarity:", cos_sim)
print("Euclidean Distance:", euc_dist)
print("Manhattan Distance:", man_dist)
print("Dot Product:", dot_product)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine Similarity: [[0.7304046]]
Euclidean Distance: 0.7342962026596069
Manhattan Distance: 11.133402
Dot Product: 0.73040456


In [8]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from typing import List, Tuple


class SimpleRAG:
    def __init__(self, embed_model: str = "all-MiniLM-L6-v2"):
        print("Loading embedding model...")

        self.embedder = SentenceTransformer(embed_model)

        self.index = None
        self.documents = []

        # all-MiniLM-L6-v2 output dimension
        self.dim = 384

    def add_documents(self, docs: List[str]):

        self.documents.extend(docs)

        print(f"Adding {len(docs)} documents | Total: {len(self.documents)}")

        embeddings = self.embedder.encode(
            docs,
            normalize_embeddings=True
        ).astype(np.float32)

        # Inner Product Index
        if self.index is None:
            self.index = faiss.IndexFlatIP(self.dim)

        self.index.add(embeddings)

        print("✅ FAISS index ready!\n")

    def retrieve(self, query: str, top_k: int = 5) -> List[Tuple[str, float]]:

        if len(self.documents) == 0:
            raise ValueError("No documents added!")

        query_vec = self.embedder.encode(
            [query],
            normalize_embeddings=True
        ).astype(np.float32)

        scores, indices = self.index.search(query_vec, top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):

            results.append(
                (self.documents[idx], float(score))
            )

        return results


def load_qwen(model_name="Qwen/Qwen2.5-7B-Instruct"):

    print(f"Loading {model_name} ...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

    print("✅ Qwen loaded successfully!")

    return tokenizer, model


def generate_answer(
    query,
    retrieved_docs,
    tokenizer,
    model,
    max_new_tokens=512
):

    context = "\n\n".join(
        [doc for doc, _ in retrieved_docs]
    )

    prompt = f"""
You are a helpful assistant.

Answer ONLY from the context.

If answer is not present, say:
"I don't have enough information."

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    answer = tokenizer.decode(
        output[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer.strip()


if __name__ == "__main__":

    rag = SimpleRAG()

    docs = [
        "Python is a popular programming language known for its simplicity and readability.",
        "Retrieval Augmented Generation (RAG) combines retrieval and generation to improve LLM answers.",
        "FAISS is a library for fast similarity search created by Facebook.",
        "Qwen2.5 is a powerful open-source LLM developed by Alibaba.",
        "Indore is a clean and modern city in Madhya Pradesh, India.",
        "Cosine similarity is commonly used for text similarity."
    ]

    rag.add_documents(docs)

    tokenizer, model = load_qwen()

    query = "What is RAG and why is it useful?"

    print(f"\n🔍 Query: {query}\n")

    results = rag.retrieve(query, top_k=4)

    print("📚 Retrieved Documents:\n")

    for i, (doc, score) in enumerate(results, 1):

        print(f"{i}. Score: {score:.4f}")
        print(doc)
        print()

    print("🤖 Generating Answer...\n")

    answer = generate_answer(
        query,
        results,
        tokenizer,
        model
    )

    print("=" * 60)
    print("✅ FINAL ANSWER:\n")
    print(answer)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Adding 6 documents | Total: 6
✅ FAISS index ready!

Loading Qwen/Qwen2.5-7B-Instruct ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Qwen loaded successfully!

🔍 Query: What is RAG and why is it useful?

📚 Retrieved Documents:

1. Score: 0.3775
Retrieval Augmented Generation (RAG) combines retrieval and generation to improve LLM answers.

2. Score: 0.1423
Indore is a clean and modern city in Madhya Pradesh, India.

3. Score: 0.1411
Cosine similarity is commonly used for text similarity.

4. Score: 0.1306
Python is a popular programming language known for its simplicity and readability.

🤖 Generating Answer...

✅ FINAL ANSWER:

RAG stands for Retrieval Augmented Generation. It is useful because it combines retrieval and generation to improve LLM answers. This method allows the model to fetch relevant information from a corpus during the generation process, enhancing the accuracy and relevance of the generated responses.


In [9]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from typing import List, Tuple


class SimpleRAG:

    def __init__(self, embed_model="all-MiniLM-L6-v2"):

        print("Loading embedding model...")

        self.embedder = SentenceTransformer(embed_model)

        self.index = None
        self.documents = []

        # Dynamic embedding dimension
        self.dim = self.embedder.get_sentence_embedding_dimension()

    def add_documents(self, docs: List[str]):

        self.documents.extend(docs)

        print(f"Adding {len(docs)} documents")

        embeddings = self.embedder.encode(
            docs,
            normalize_embeddings=True
        ).astype(np.float32)

        if self.index is None:
            self.index = faiss.IndexFlatIP(self.dim)

        self.index.add(embeddings)

        print("✅ FAISS index ready!\n")

    def retrieve(
        self,
        query: str,
        top_k: int = 5
    ) -> List[Tuple[str, float]]:

        if len(self.documents) == 0:
            raise ValueError("No documents added!")

        query_vec = self.embedder.encode(
            [query],
            normalize_embeddings=True
        ).astype(np.float32)

        scores, indices = self.index.search(query_vec, top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):

            results.append(
                (self.documents[idx], float(score))
            )

        return results


def load_mistral(
    model_name="mistralai/Mistral-7B-Instruct-v0.3"
):

    print(f"Loading {model_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    print("✅ Mistral loaded successfully!")

    return tokenizer, model


def generate_answer(
    query,
    retrieved_docs,
    tokenizer,
    model,
    max_new_tokens=256
):

    context = "\n\n".join(
        [doc for doc, _ in retrieved_docs]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. "
                "Answer ONLY from the given context. "
                "If answer is not available, say "
                "'I don't have enough information.'"
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{query}
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        output[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer.strip()


if __name__ == "__main__":

    rag = SimpleRAG()

    docs = [
        "Python is a popular programming language known for its simplicity and readability.",

        "Retrieval Augmented Generation (RAG) combines retrieval and generation to improve LLM answers.",

        "FAISS is a library for fast similarity search created by Facebook.",

        "Mistral is a powerful open-source LLM known for its strong performance.",

        "Indore is a clean and modern city in Madhya Pradesh, India.",

        "Cosine similarity is widely used for measuring text similarity."
    ]

    rag.add_documents(docs)

    tokenizer, model = load_mistral()

    query = "What is RAG and why is it useful?"

    print(f"\n🔍 Query: {query}\n")

    results = rag.retrieve(query, top_k=4)

    print("📚 Retrieved Documents:\n")

    for i, (doc, score) in enumerate(results, 1):

        print(f"{i}. Score: {score:.4f}")
        print(doc)
        print()

    print("🤖 Generating Answer...\n")

    answer = generate_answer(
        query,
        results,
        tokenizer,
        model
    )

    print("=" * 60)
    print("✅ FINAL ANSWER:\n")
    print(answer)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Adding 6 documents
✅ FAISS index ready!

Loading mistralai/Mistral-7B-Instruct-v0.3...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Mistral loaded successfully!

🔍 Query: What is RAG and why is it useful?

📚 Retrieved Documents:

1. Score: 0.3775
Retrieval Augmented Generation (RAG) combines retrieval and generation to improve LLM answers.

2. Score: 0.1423
Indore is a clean and modern city in Madhya Pradesh, India.

3. Score: 0.1306
Python is a popular programming language known for its simplicity and readability.

4. Score: 0.1294
Mistral is a powerful open-source LLM known for its strong performance.

🤖 Generating Answer...

✅ FINAL ANSWER:

RAG stands for Retrieval Augmented Generation. It is a method that combines retrieval and generation to improve the answers provided by Language Model Machines (LLMs). This approach can help enhance the accuracy and completeness of responses by incorporating relevant external data during the response generation process. Therefore, it is useful for providing more accurate and informative answers.


In [10]:
# Run this first — installs everything you need
!pip install sentence-transformers faiss-cpu chromadb
!pip install pypdf ragas rank_bm25 google-generativeai streamlit
!pip install transformers pandas numpy arxiv openai tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    

In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    'What is RAG?',
    'RAG stands for Retrieval Augmented Generation.',
    'I love playing cricket.',
]

embeddings = model.encode(sentences)

print('Shape of embeddings:', embeddings.shape)
# Output: (3, 384) — 3 sentences, each with 384 numbers

print('\nFirst 5 numbers of sentence 1 embedding:')
print(embeddings[0][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of embeddings: (3, 384)

First 5 numbers of sentence 1 embedding:
[-0.06957071  0.09519999  0.01602142  0.00680149 -0.088405  ]


In [12]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

sentence1 = 'What is machine learning?'
sentence2 = 'Explain ML to me'
sentence3 = 'I like eating mangoes'

emb1 = model.encode(sentence1)
emb2 = model.encode(sentence2)
emb3 = model.encode(sentence3)

# Calculate similarity scores (0 = different, 1 = same)
score_12 = util.cos_sim(emb1, emb2)
score_13 = util.cos_sim(emb1, emb3)

print(f'Similarity (ML questions): {score_12.item():.2f}')  # Should be HIGH
print(f'Similarity (ML vs mango): {score_13.item():.2f}')  # Should be LOW

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity (ML questions): 0.48
Similarity (ML vs mango): 0.09


In [13]:
def recursive_text_splitter(text, chunk_size=100, chunk_overlap=20):
    """Simple recursive-style text splitter by characters"""
    if not text:
        return []
    
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        sentence_len = len(sentence)
        
        if sentence_len > chunk_size:
            words = sentence.split()
            for i in range(0, len(words), chunk_size - chunk_overlap):
                chunk_words = words[i:i + chunk_size]
                chunk = " ".join(chunk_words)
                chunks.append(chunk)
            continue
        
        if current_length + sentence_len + 1 <= chunk_size:
            current_chunk.append(sentence)
            current_length += sentence_len + 1
        else:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_length = sentence_len
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    final_chunks = []
    for i, chunk in enumerate(chunks):
        final_chunks.append(chunk)
    
    return final_chunks

long_text = """
Artificial Intelligence is transforming the world. Machine learning is a subset of AI.
Deep learning uses neural networks with many layers. Natural language processing helps
computers understand human language. Large language models like GPT are trained on
billions of words. RAG systems combine retrieval with generation. Vector databases
store embeddings efficiently. Semantic search finds relevant documents by meaning.
Fine-tuning adapts models to specific tasks. Prompt engineering improves AI responses.
"""

chunks = recursive_text_splitter(long_text, chunk_size=100, chunk_overlap=20)

print(f'Total chunks created: {len(chunks)}\n')
for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ---')
    print(chunk)
    print(f'Length: {len(chunk)} characters\n')

Total chunks created: 7

--- Chunk 1 ---
Artificial Intelligence is transforming the world. Machine learning is a subset of AI.
Length: 86 characters

--- Chunk 2 ---
Deep learning uses neural networks with many layers.
Length: 52 characters

--- Chunk 3 ---
Natural language processing helps
computers understand human language.
Length: 70 characters

--- Chunk 4 ---
Large language models like GPT are trained on
billions of words.
Length: 64 characters

--- Chunk 5 ---
RAG systems combine retrieval with generation. Vector databases
store embeddings efficiently.
Length: 93 characters

--- Chunk 6 ---
Semantic search finds relevant documents by meaning. Fine-tuning adapts models to specific tasks.
Length: 97 characters

--- Chunk 7 ---
Prompt engineering improves AI responses.
Length: 41 characters



In [14]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Our knowledge base 
documents = [
    'RAG stands for Retrieval Augmented Generation.',
    'FAISS is a library for efficient similarity search.',
    'LangChain helps build LLM-powered applications.',
    'Embeddings convert text into numerical vectors.',
    'ChromaDB is an open-source vector database.',
]

# Convert documents to embeddings
embeddings = model.encode(documents)
embeddings = np.array(embeddings).astype('float32')

# Build FAISS index
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f'Total documents in index: {index.ntotal}')

# Search for a query
query = 'What is a vector database?'
query_embedding = model.encode([query]).astype('float32')

# Get top 2 results
distances, indices = index.search(query_embedding, k=2)

print(f'\nQuery: {query}')
print('\nTop results:')
for i, idx in enumerate(indices[0]):
    print(f'{i+1}. {documents[idx]} (distance: {distances[0][i]:.2f})')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total documents in index: 5

Query: What is a vector database?

Top results:
1. ChromaDB is an open-source vector database. (distance: 0.61)
2. Embeddings convert text into numerical vectors. (distance: 1.24)


In [15]:
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp

!pip install chromadb==0.5.5
!pip install opentelemetry-api==1.24.0
!pip install opentelemetry-sdk==1.24.0
!pip install opentelemetry-exporter-otlp==1.24.0

Found existing installation: opentelemetry-api 1.41.1
Uninstalling opentelemetry-api-1.41.1:
  Successfully uninstalled opentelemetry-api-1.41.1
Found existing installation: opentelemetry-sdk 1.41.1
Uninstalling opentelemetry-sdk-1.41.1:
  Successfully uninstalled opentelemetry-sdk-1.41.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
  Using cached opentelemetry_api-1.41.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.41.1-py3-none-any.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 13.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.9 MB/s eta 0:00:00:00:0100:01
Using cached opentelemetry_api-1.41.1-py3-none-any.whl (69 kB)
Using cached opentelemetry_sdk-1.41.1-py3-none-any.whl (180 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.8/240.8 kB 14.8 MB/s eta 0:00:00
  Attempting 

In [16]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Create ChromaDB client 
client = chromadb.PersistentClient(path='./my_chroma_db')

# Create or get a collection
collection = client.get_or_create_collection(name='rag_workshop')

# Our documents
documents = [
    'Python is a popular programming language.',
    'RAG improves LLM accuracy with external knowledge.',
    'Streamlit makes it easy to build web apps.',
    'Google Gemini is a powerful AI model.',
]

# Add documents to ChromaDB
collection.add(
    documents=documents,
    ids=[f'doc_{i}' for i in range(len(documents))]
)

# Query the collection
results = collection.query(
    query_texts=['Tell me about AI models'],
    n_results=2
)

print('Query: Tell me about AI models')
print('\nResults:')
for doc in results['documents'][0]:
    print('-', doc)

AttributeError: `np.float_` was removed in the NumPy 2.0 release. Use `np.float64` instead.

In [17]:
import re
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

with open('sample_doc.txt', 'w', encoding='utf-8') as f:
    f.write("""
RAG Workshop Guide

Chapter 1: Introduction to RAG
RAG stands for Retrieval Augmented Generation. It combines information retrieval 
with language generation. This allows AI to answer questions using your own documents instead of only relying on trained knowledge.

Chapter 2: How Embeddings Work
Embeddings are numerical representations of text. Similar texts have similar embeddings.
We use models like sentence-transformers to create embeddings.

Chapter 3: Vector Databases
Vector databases store embeddings and allow fast similarity search using cosine similarity or Euclidean distance.
Popular ones are FAISS, ChromaDB, and Pinecone.
""")

print("✅ Document created!\n")

with open('sample_doc.txt', 'r', encoding='utf-8') as f:
    text = f.read()

def split_into_chunks(text, chunk_size=250, chunk_overlap=50):
    # Split by double newlines (paragraphs)
    paragraphs = re.split(r'\n\s*\n', text.strip())
    chunks = []
    current_chunk = []
    current_len = 0
    
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
            
        if current_len + len(para) > chunk_size and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            # Overlap: keep last part for next chunk
            current_chunk = current_chunk[-1:] + [para]
            current_len = sum(len(p) for p in current_chunk)
        else:
            current_chunk.append(para)
            current_len += len(para)
    
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    
    return chunks

chunks = split_into_chunks(text)
print(f"✅ Created {len(chunks)} chunks\n")

print("Loading embedding model (this may take a few seconds first time)...")
model = SentenceTransformer('all-MiniLM-L6-v2')   # Small, fast & good quality
print("Model loaded successfully!\n")

print("Creating embeddings...")
embeddings = model.encode(chunks, convert_to_numpy=True)
print(f"✅ Embeddings created with shape: {embeddings.shape}\n")

def rag_query(question, top_k=2):
    
    query_embedding = model.encode([question], convert_to_numpy=True)
    
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    print(f"🔍 Question: {question}\n")
    print("📄 Retrieved Context:\n")
    
    retrieved_context = ""
    for rank, idx in enumerate(top_indices, 1):
        score = similarities[idx]
        print(f"[{rank}] Similarity: {score:.4f}")
        print(chunks[idx])
        print("-" * 80)
        retrieved_context += f"Context {rank}:\n" + chunks[idx] + "\n\n"
    
    print("🤖 Answer (RAG):")
    print("In a real system, you would send this prompt to an LLM:")
    print(f"""
Context:
{retrieved_context}
Question: {question}
Answer:""")
    
    print("→ (Answer would be generated here using the retrieved context)")

rag_query("What is RAG?")
print("\n" + "="*70 + "\n")
rag_query("What are popular vector databases?")
print("\n" + "="*70 + "\n")
rag_query("How do embeddings work?")

✅ Document created!

✅ Created 4 chunks

Loading embedding model (this may take a few seconds first time)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!

Creating embeddings...
✅ Embeddings created with shape: (4, 384)

🔍 Question: What is RAG?

📄 Retrieved Context:

[1] Similarity: 0.6786
RAG Workshop Guide
--------------------------------------------------------------------------------
[2] Similarity: 0.5733
RAG Workshop Guide

Chapter 1: Introduction to RAG
RAG stands for Retrieval Augmented Generation. It combines information retrieval 
with language generation. This allows AI to answer questions using your own documents instead of only relying on trained knowledge.
--------------------------------------------------------------------------------
🤖 Answer (RAG):
In a real system, you would send this prompt to an LLM:

Context:
Context 1:
RAG Workshop Guide

Context 2:
RAG Workshop Guide

Chapter 1: Introduction to RAG
RAG stands for Retrieval Augmented Generation. It combines information retrieval 
with language generation. This allows AI to answer questions using your own documents instead of only relying

In [18]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

corpus = [
    'RAG uses retrieval to improve AI answers.',
    'FAISS is used for fast vector search.',
    'Python is widely used for machine learning.',
    'LangChain connects LLMs with external tools.',
    'Gemini is Google\'s multimodal AI model.',
]

query = 'vector search library'

# --- BM25 Keyword Search ---
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(query.lower().split())

# --- Vector Search ---
corpus_embeddings = model.encode(corpus)
query_embedding = model.encode(query)
vector_scores = util.cos_sim(query_embedding, corpus_embeddings)[0].numpy()

# --- Combine scores (simple average) ---
bm25_norm = bm25_scores / (bm25_scores.max() + 1e-9)
vector_norm = (vector_scores - vector_scores.min()) / (vector_scores.max() - vector_scores.min() + 1e-9)
combined = 0.5 * bm25_norm + 0.5 * vector_norm

# Get top results
top_indices = combined.argsort()[::-1][:3]

print(f'Query: "{query}"')
print('\nTop results (Hybrid Search):')
for i, idx in enumerate(top_indices):
    print(f'{i+1}. [{combined[idx]:.2f}] {corpus[idx]}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: "vector search library"

Top results (Hybrid Search):
1. [1.00] FAISS is used for fast vector search.
2. [0.26] Python is widely used for machine learning.
3. [0.24] Gemini is Google's multimodal AI model.


In [1]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

knowledge_base = [
    'RAG stands for Retrieval Augmented Generation.',
    'FAISS is a fast vector similarity search library.',
    'Embeddings are dense vector representations.',
    'LangChain builds LLM-powered pipelines.',
    'ChromaDB is an open-source vector store.',
]

kb_embeddings = model.encode(knowledge_base)

def normal_search(query):
    q_emb = model.encode(query)
    scores = util.cos_sim(q_emb, kb_embeddings)[0]
    best = scores.argmax().item()
    return knowledge_base[best], scores[best].item()

def hyde_search(query):
    # Step 1: Generate a hypothetical answer 
    hypothetical_answer = f'The answer to "{query}" involves vector similarity and retrieval methods.'
    
    # Step 2: Embed the hypothetical answer (not the query)
    hyde_emb = model.encode(hypothetical_answer)
    scores = util.cos_sim(hyde_emb, kb_embeddings)[0]
    best = scores.argmax().item()
    return knowledge_base[best], scores[best].item()

query = 'How does vector retrieval work?'

normal_result, normal_score = normal_search(query)
hyde_result, hyde_score = hyde_search(query)

print(f'Query: {query}')
print(f'\nNormal Search  [{normal_score:.2f}]: {normal_result}')
print(f'HyDE Search    [{hyde_score:.2f}]: {hyde_result}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: How does vector retrieval work?

Normal Search  [0.50]: FAISS is a fast vector similarity search library.
HyDE Search    [0.58]: FAISS is a fast vector similarity search library.


In [2]:
import os
from sentence_transformers import SentenceTransformer, util
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Simulate multiple documents 
documents = {
    'doc1.txt': 'Artificial Intelligence is transforming industries. AI includes ML and deep learning.',
    'doc2.txt': 'RAG systems retrieve relevant documents before generating answers with LLMs.',
    'doc3.txt': 'Python is the most popular language for data science and AI development.',
    'doc4.txt': 'Vector databases like FAISS and ChromaDB store text embeddings efficiently.',
    'doc5.txt': 'Streamlit allows you to build interactive web apps using only Python code.',
}

# Build index from all documents
all_chunks = []
all_sources = []

for filename, content in documents.items():
    # Simple chunking by sentence
    sentences = content.split('. ')
    for s in sentences:
        if s.strip():
            all_chunks.append(s.strip())
            all_sources.append(filename)

# Embed all chunks
chunk_embeddings = model.encode(all_chunks)

def multi_doc_search(query, top_k=3):
    q_emb = model.encode(query)
    scores = util.cos_sim(q_emb, chunk_embeddings)[0]
    top_indices = scores.argsort(descending=True)[:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'source': all_sources[idx],
            'text': all_chunks[idx],
            'score': scores[idx].item()
        })
    return results

query = 'What are vector databases?'
results = multi_doc_search(query)

print(f'Query: {query}\n')
for r in results:
    print(f'Source: {r["source"]} | Score: {r["score"]:.2f}')
    print(f'Text: {r["text"]}\n')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: What are vector databases?

Source: doc4.txt | Score: 0.53
Text: Vector databases like FAISS and ChromaDB store text embeddings efficiently.

Source: doc1.txt | Score: 0.27
Text: AI includes ML and deep learning.

Source: doc3.txt | Score: 0.22
Text: Python is the most popular language for data science and AI development.



In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

# Create sample FAQ CSV 
faq_data = {
    'question': [
        'What is RAG?',
        'How do embeddings work?',
        'What is FAISS?',
        'What is LangChain?',
        'What is Streamlit?',
    ],
    'answer': [
        'RAG is Retrieval Augmented Generation — it lets AI use your documents to answer questions.',
        'Embeddings convert text into numbers so computers can understand meaning.',
        'FAISS is a fast similarity search library developed by Facebook AI.',
        'LangChain is a framework for building AI applications powered by language models.',
        'Streamlit is a Python library for building web applications quickly.',
    ]
}

df = pd.DataFrame(faq_data)
df.to_csv('faq.csv', index=False)

# Load the FAQ
df = pd.read_csv('faq.csv')

# Embed all questions
question_embeddings = model.encode(df['question'].tolist())

def faq_chatbot(user_query, threshold=0.4):
    q_emb = model.encode(user_query)
    scores = util.cos_sim(q_emb, question_embeddings)[0]
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()
    
    if best_score >= threshold:
        return df['answer'][best_idx], df['question'][best_idx], best_score
    else:
        return 'Sorry, I could not find a relevant answer.', None, best_score

# Test the chatbot
test_queries = ['explain RAG to me', 'what is langchain used for', 'how to build web app']

for q in test_queries:
    answer, matched_q, score = faq_chatbot(q)
    print(f'User: {q}')
    print(f'Matched FAQ: {matched_q} (score: {score:.2f})')
    print(f'Answer: {answer}\n')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


User: explain RAG to me
Matched FAQ: What is RAG? (score: 0.92)
Answer: RAG is Retrieval Augmented Generation — it lets AI use your documents to answer questions.

User: what is langchain used for
Matched FAQ: What is LangChain? (score: 0.86)
Answer: LangChain is a framework for building AI applications powered by language models.

User: how to build web app
Matched FAQ: None (score: 0.13)
Answer: Sorry, I could not find a relevant answer.



In [4]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

model_names = [
    'all-MiniLM-L6-v2',        # Fast, small
    'all-mpnet-base-v2',       # More accurate, slower
    'paraphrase-MiniLM-L3-v2'  # Fastest, smallest
]

query = 'What is machine learning?'
positive = 'Machine learning is a subset of AI that learns from data.'
negative = 'The weather today is sunny and warm.'

results = []

for name in model_names:
    model = SentenceTransformer(name)
    
    q_emb = model.encode(query)
    p_emb = model.encode(positive)
    n_emb = model.encode(negative)
    
    pos_score = util.cos_sim(q_emb, p_emb).item()
    neg_score = util.cos_sim(q_emb, n_emb).item()
    
    results.append({
        'Model': name,
        'Positive Score': round(pos_score, 3),
        'Negative Score': round(neg_score, 3),
        'Gap (higher=better)': round(pos_score - neg_score, 3)
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

                  Model  Positive Score  Negative Score  Gap (higher=better)
       all-MiniLM-L6-v2           0.796           0.050                0.747
      all-mpnet-base-v2           0.811           0.023                0.788
paraphrase-MiniLM-L3-v2           0.748          -0.045                0.794


In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample legal clauses
legal_clauses = [
    'Section 1 - Termination: Either party may terminate this agreement with 30 days written notice.',
    'Section 2 - Confidentiality: All shared information must be kept strictly confidential for 5 years.',
    'Section 3 - Payment: Invoices must be paid within 15 business days of receipt.',
    'Section 4 - Liability: Neither party shall be liable for indirect or consequential damages.',
    'Section 5 - Governing Law: This agreement is governed by the laws of the State of California.',
    'Section 6 - Intellectual Property: All work produced belongs exclusively to the client.',
    'Section 7 - Dispute Resolution: Disputes shall be resolved through binding arbitration.',
]

clause_embeddings = model.encode(legal_clauses)

def find_relevant_clause(scenario):
    s_emb = model.encode(scenario)
    scores = util.cos_sim(s_emb, clause_embeddings)[0]
    top3 = scores.argsort(descending=True)[:3]
    
    print(f'Scenario: {scenario}')
    print('Relevant clauses:')
    for idx in top3:
        print(f'  Score {scores[idx]:.2f}: {legal_clauses[idx][:80]}...')
    print()

find_relevant_clause('I want to end the contract early')
find_relevant_clause('Who owns the code I write for the client?')
find_relevant_clause('When must I pay the invoice?')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scenario: I want to end the contract early
Relevant clauses:
  Score 0.48: Section 1 - Termination: Either party may terminate this agreement with 30 days ...
  Score 0.23: Section 3 - Payment: Invoices must be paid within 15 business days of receipt....
  Score 0.15: Section 7 - Dispute Resolution: Disputes shall be resolved through binding arbit...

Scenario: Who owns the code I write for the client?
Relevant clauses:
  Score 0.45: Section 6 - Intellectual Property: All work produced belongs exclusively to the ...
  Score 0.15: Section 5 - Governing Law: This agreement is governed by the laws of the State o...
  Score 0.14: Section 4 - Liability: Neither party shall be liable for indirect or consequenti...

Scenario: When must I pay the invoice?
Relevant clauses:
  Score 0.69: Section 3 - Payment: Invoices must be paid within 15 business days of receipt....
  Score 0.28: Section 1 - Termination: Either party may terminate this agreement with 30 days ...
  Score 0.21: Section 4 - Liab

In [6]:
from sentence_transformers import SentenceTransformer, util

# Multilingual model
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Mixed language knowledge base
knowledge_base = [
    'RAG stands for Retrieval Augmented Generation.',          # English
    'RAG ka matlab hai Retrieval Augmented Generation.',       # Hindi transliteration
    'Embeddings text ko numbers mein convert karte hain.',     # Hinglish
    'Vector databases store information as numerical arrays.', # English
    'LangChain ek powerful AI framework hai.',                 # Hinglish
]

kb_embeddings = model.encode(knowledge_base)

def multilingual_search(query):
    q_emb = model.encode(query)
    scores = util.cos_sim(q_emb, kb_embeddings)[0]
    top2 = scores.argsort(descending=True)[:2]
    
    print(f'Query: "{query}"')
    for idx in top2:
        print(f'  [{scores[idx]:.2f}] {knowledge_base[idx]}')
    print()

multilingual_search('What is RAG?')               # English query
multilingual_search('RAG kya hota hai?')           # Hindi query
multilingual_search('embeddings kaise kaam karte hain')  # Hinglish

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: "What is RAG?"
  [0.69] RAG stands for Retrieval Augmented Generation.
  [0.37] RAG ka matlab hai Retrieval Augmented Generation.

Query: "RAG kya hota hai?"
  [0.36] Embeddings text ko numbers mein convert karte hain.
  [0.35] RAG stands for Retrieval Augmented Generation.

Query: "embeddings kaise kaam karte hain"
  [0.59] Embeddings text ko numbers mein convert karte hain.
  [0.25] LangChain ek powerful AI framework hai.



In [7]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

def check_hallucination(context, answer, threshold=0.4):
    """
    Checks if the answer is supported by the context.
    Low similarity = likely hallucination.
    """
    ctx_emb = model.encode(context)
    ans_emb = model.encode(answer)
    score = util.cos_sim(ctx_emb, ans_emb).item()
    
    label = 'Faithful' if score >= threshold else 'Possible Hallucination'
    return score, label

# Test cases
context = 'FAISS is a library developed by Facebook AI for fast similarity search of dense vectors.'

test_answers = [
    ('FAISS is a fast similarity search tool made by Facebook.', 'Good answer'),
    ('FAISS was created by Google for image recognition tasks.', 'Hallucinated answer'),
    ('I enjoy playing cricket on weekends.', 'Completely off-topic'),
]

print(f'Context: "{context}"\n')
for answer, label in test_answers:
    score, verdict = check_hallucination(context, answer)
    print(f'Answer: "{answer[:60]}..."')
    print(f'Expected: {label} | Score: {score:.2f} | Verdict: {verdict}\n')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Context: "FAISS is a library developed by Facebook AI for fast similarity search of dense vectors."

Answer: "FAISS is a fast similarity search tool made by Facebook...."
Expected: Good answer | Score: 0.84 | Verdict: Faithful

Answer: "FAISS was created by Google for image recognition tasks...."
Expected: Hallucinated answer | Score: 0.55 | Verdict: Faithful

Answer: "I enjoy playing cricket on weekends...."
Expected: Completely off-topic | Score: -0.01 | Verdict: Possible Hallucination

